In [0]:
# Paste your copied workspace file path here
path = "/Workspace/Users/azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com/bookstore_orders_databricks.csv"

df = spark.read.csv(
    "file:" + path,
    header=True,
    inferSchema=True
)

display(df)

order_id,city,book_category,quantity,unit_price,order_status
1001,Hyderabad,Technology,2,850,Completed
1002,Bengaluru,Fiction,3,400,Completed
1003,Chennai,Science,1,650,Completed
1004,Hyderabad,Fiction,5,350,Completed
1005,Pune,History,2,500,Cancelled
1006,Bengaluru,Technology,1,1200,Completed
1007,Chennai,Fiction,4,300,Completed
1008,Pune,Science,2,750,Completed
1009,Hyderabad,History,3,550,Completed
1010,Bengaluru,Science,2,700,Cancelled


EXERCISES


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Update paths with your Unity Catalog volume credentials
csv_path = "/Workspace/Users/azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com/cinema_bookings.csv"
json_path = "/Workspace/Users/azuser7408_mml.local@karthikirisoutlook.onmicrosoft.com/cinema_master.json"

bookings_df = spark.read.csv(csv_path, header=True, inferSchema=True)
cinemas_df = spark.read.json(json_path)

In [0]:
bookings_df = spark.read.csv(csv_path, header=True, inferSchema=True)

In [0]:

cinemas_df = spark.read.json(json_path)

display(bookings_df)
display(cinemas_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash


cinema_id,cinema_name,facilities,location,screen_count
C001,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C002,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7
C003,Metro Movies,"List(Food Court, Wheelchair Access)","List(Mumbai, Central)",4
C004,Coastal Cinema,List(Parking),"List(Chennai, South)",3
C005,Capital Cineplex,"List(Parking, Food Court, Wheelchair Access)","List(Delhi, East)",8
C006,Orchid Screens,"List(Food Court, Wheelchair Access)","List(Pune, West)",4


In [0]:
print("Bookings Schema")
bookings_df.printSchema()

print("Cinema Master Schema")
cinemas_df.printSchema()

--- Bookings Schema ---
root
 |-- booking_id: string (nullable = true)
 |-- cinema_id: string (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- show_date: date (nullable = true)
 |-- tickets: integer (nullable = true)
 |-- price_per_ticket: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_method: string (nullable = true)

--- Cinema Master Schema ---
root
 |-- cinema_id: string (nullable = true)
 |-- cinema_name: string (nullable = true)
 |-- facilities: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zone: string (nullable = true)
 |-- screen_count: long (nullable = true)



In [0]:
display(bookings_df.limit(10))
total_rows = bookings_df.count()
print(f"Total CSV rows: {total_rows}")

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash


Total CSV rows: 32


In [0]:
selected_df = bookings_df.select("booking_id", "movie_title", "tickets", "booking_status")
display(selected_df)

booking_id,movie_title,tickets,booking_status
B1001,Starlight Express,2,Confirmed
B1002,The Last Island,3,Confirmed
B1003,Code Red,1,Cancelled
B1004,River of Dreams,4,Confirmed
B1005,Starlight Express,2,Confirmed
B1006,The Last Island,5,Confirmed
B1007,Code Red,3,Confirmed
B1008,River of Dreams,2,Refunded
B1009,Starlight Express,4,Confirmed
B1010,The Last Island,1,Confirmed


SECTION B

In [0]:
confirmed_df = bookings_df.filter(F.col("booking_status") == "Confirmed")
display(confirmed_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash
B1011,C005,Code Red,2026-10-06,2026-10-08,2,320,null,Confirmed,Card
B1013,C001,The Last Island,2026-10-07,2026-10-09,5,250,10,Confirmed,Card


In [0]:
high_value_df = bookings_df.filter(
    (F.col("tickets") > 3) & (F.col("price_per_ticket") > 200)
)
display(high_value_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1013,C001,The Last Island,2026-10-07,2026-10-09,5,250,10,Confirmed,Card
B1017,C005,The Last Island,2026-10-09,2026-10-11,4,320,15,Refunded,Card
B1023,C005,River of Dreams,2026-10-12,2026-10-14,5,320,10,Confirmed,Card
B1029,C005,Starlight Express,2026-10-15,2026-10-17,4,320,5,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,null,Confirmed,UPI


In [0]:
null_counts = bookings_df.select(
    F.count(F.when(F.col("discount_pct").isNull(), 1)).alias("missing_discount_pct"),
    F.count(F.when(F.col("payment_method").isNull(), 1)).alias("missing_payment_method")
)
display(null_counts)

missing_discount_pct,missing_payment_method
5,2


In [0]:
cleaned_bookings_df = bookings_df.fillna({
    "discount_pct": 0,
    "payment_method": "Unknown"
})
display(cleaned_bookings_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash


In [0]:
bookings_gross_df = cleaned_bookings_df.withColumn(
    "gross_amount", 
    F.col("tickets") * F.col("price_per_ticket")
)
display(bookings_gross_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI,500
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card,900
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI,220
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash,720
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card,640
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI,750
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card,600
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI,880
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash,180


In [0]:
bookings_net_df = bookings_gross_df.withColumn(
    "net_amount",
    F.round(F.col("gross_amount") * (1 - (F.col("discount_pct") / 100)), 2)
)
display(bookings_net_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,net_amount
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI,500,450.0
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card,900,900.0
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI,220,209.0
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash,720,720.0
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card,640,544.0
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050,1050.0
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI,750,750.0
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card,600,540.0
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI,880,836.0
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash,180,180.0


In [0]:
bookings_sized_df = bookings_net_df.withColumn(
    "booking_size",
    F.when(F.col("tickets") >= 4, "Large").otherwise("Regular")
)
display(bookings_sized_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,net_amount,booking_size
B1001,C001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI,500,450.0,Regular
B1002,C002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card,900,900.0,Regular
B1003,C003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI,220,209.0,Regular
B1004,C004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash,720,720.0,Large
B1005,C005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card,640,544.0,Regular
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050,1050.0,Large
B1007,C001,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI,750,750.0,Regular
B1008,C002,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card,600,540.0,Regular
B1009,C003,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI,880,836.0,Large
B1010,C004,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash,180,180.0,Regular


In [0]:
top_5_net_df = bookings_sized_df.orderBy(F.col("net_amount").desc()).limit(5)
display(top_5_net_df)

booking_id,cinema_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,net_amount,booking_size
B1023,C005,River of Dreams,2026-10-12,2026-10-14,5,320,10,Confirmed,Card,1600,1440.0,Large
B1029,C005,Starlight Express,2026-10-15,2026-10-17,4,320,5,Confirmed,Card,1280,1216.0,Large
B1013,C001,The Last Island,2026-10-07,2026-10-09,5,250,10,Confirmed,Card,1250,1125.0,Large
B1017,C005,The Last Island,2026-10-09,2026-10-11,4,320,15,Refunded,Card,1280,1088.0,Large
B1006,C006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050,1050.0,Large


SECTION C

In [0]:
status_counts_df = bookings_sized_df.groupBy("booking_status").count()
display(status_counts_df)

booking_status,count
Refunded,3
Confirmed,25
Cancelled,4


In [0]:

movie_summary_df = bookings_sized_df.filter(F.col("booking_status") == "Confirmed") \
    .groupBy("movie_title").agg(
        F.sum("tickets").alias("total_tickets"),
        F.round(F.sum("net_amount"), 2).alias("total_net_revenue")
    )
display(movie_summary_df)

movie_title,total_tickets,total_net_revenue
River of Dreams,10,2380.0
The Last Island,26,5905.0
Code Red,13,3087.0
Starlight Express,23,5500.0


In [0]:
top_cinemas_df = bookings_sized_df.filter(F.col("booking_status") == "Confirmed").groupBy("cinema_id").agg(
    F.round(F.sum("net_amount"), 2).alias("confirmed_net_revenue")).orderBy(
    F.col("confirmed_net_revenue").desc()).limit(3)
display(top_cinemas_df)

cinema_id,confirmed_net_revenue
C005,3840.0
C001,3075.0
C002,2610.0


In [0]:
joined_df = bookings_sized_df.join(cinemas_df, on="cinema_id", how="left")
display(joined_df)

cinema_id,booking_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,net_amount,booking_size,cinema_name,facilities,location,screen_count
C001,B1001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI,500,450.0,Regular,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C002,B1002,The Last Island,2026-10-01,2026-10-02,3,300,0,Confirmed,Card,900,900.0,Regular,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7
C003,B1003,Code Red,2026-10-02,2026-10-04,1,220,5,Cancelled,UPI,220,209.0,Regular,Metro Movies,"List(Food Court, Wheelchair Access)","List(Mumbai, Central)",4
C004,B1004,River of Dreams,2026-10-02,2026-10-05,4,180,0,Confirmed,Cash,720,720.0,Large,Coastal Cinema,List(Parking),"List(Chennai, South)",3
C005,B1005,Starlight Express,2026-10-03,2026-10-05,2,320,15,Confirmed,Card,640,544.0,Regular,Capital Cineplex,"List(Parking, Food Court, Wheelchair Access)","List(Delhi, East)",8
C006,B1006,The Last Island,2026-10-03,2026-10-04,5,210,0,Confirmed,UPI,1050,1050.0,Large,Orchid Screens,"List(Food Court, Wheelchair Access)","List(Pune, West)",4
C001,B1007,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI,750,750.0,Regular,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C002,B1008,River of Dreams,2026-10-04,2026-10-06,2,300,10,Refunded,Card,600,540.0,Regular,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7
C003,B1009,Starlight Express,2026-10-05,2026-10-07,4,220,5,Confirmed,UPI,880,836.0,Large,Metro Movies,"List(Food Court, Wheelchair Access)","List(Mumbai, Central)",4
C004,B1010,The Last Island,2026-10-05,2026-10-08,1,180,0,Confirmed,Cash,180,180.0,Regular,Coastal Cinema,List(Parking),"List(Chennai, South)",3


In [0]:
display(joined_df.select("cinema_name", "location.city", "movie_title", "booking_status"))

cinema_name,city,movie_title,booking_status
Galaxy Screens,Hyderabad,Starlight Express,Confirmed
Silverlight Multiplex,Bengaluru,The Last Island,Confirmed
Metro Movies,Mumbai,Code Red,Cancelled
Coastal Cinema,Chennai,River of Dreams,Confirmed
Capital Cineplex,Delhi,Starlight Express,Confirmed
Orchid Screens,Pune,The Last Island,Confirmed
Galaxy Screens,Hyderabad,Code Red,Confirmed
Silverlight Multiplex,Bengaluru,River of Dreams,Refunded
Metro Movies,Mumbai,Starlight Express,Confirmed
Coastal Cinema,Chennai,The Last Island,Confirmed


In [0]:
city_revenue_df = joined_df.filter(F.col("booking_status") == "Confirmed") \
    .groupBy(F.coalesce(F.col("location.city"), F.lit("Unknown / Unmatched")).alias("city")) \
    .agg(F.round(F.sum("net_amount"), 2).alias("confirmed_net_revenue"))
display(city_revenue_df)
unmatched_bookings_df = joined_df.filter(F.col("cinema_name").isNull())
display(unmatched_bookings_df)

city,confirmed_net_revenue
Delhi,3840.0
Chennai,2097.0
Hyderabad,3075.0
Pune,2520.0
Mumbai,2310.0
Bengaluru,2610.0
Unknown / Unmatched,420.0


cinema_id,booking_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,net_amount,booking_size,cinema_name,facilities,location,screen_count
C999,B1030,The Last Island,2026-10-15,2026-10-18,2,210,0,Confirmed,UPI,420,420.0,Regular,null,null,null,null


SECTION D

In [0]:
cinema_zones_df = cinemas_df.select("cinema_id", "cinema_name", "location.zone")
display(cinema_zones_df)

cinema_id,cinema_name,zone
C001,Galaxy Screens,West
C002,Silverlight Multiplex,North
C003,Metro Movies,Central
C004,Coastal Cinema,South
C005,Capital Cineplex,East
C006,Orchid Screens,West


In [0]:
exploded_facilities_df = cinemas_df.select(
    "cinema_id", 
    "cinema_name", 
    F.explode("facilities").alias("facility")
)
display(exploded_facilities_df)

cinema_id,cinema_name,facility
C001,Galaxy Screens,Parking
C001,Galaxy Screens,Food Court
C001,Galaxy Screens,Wheelchair Access
C002,Silverlight Multiplex,Parking
C002,Silverlight Multiplex,Food Court
C003,Metro Movies,Food Court
C003,Metro Movies,Wheelchair Access
C004,Coastal Cinema,Parking
C005,Capital Cineplex,Parking
C005,Capital Cineplex,Food Court


In [0]:
dated_bookings_df = bookings_sized_df \
    .withColumn("booking_date", F.to_date("booking_date")) \
    .withColumn("show_date", F.to_date("show_date")) \
    .withColumn("days_between_booking_and_show", F.datediff(F.col("show_date"), F.col("booking_date")))
display(dated_bookings_df.select("booking_id", "booking_date", "show_date", "days_between_booking_and_show"))

booking_id,booking_date,show_date,days_between_booking_and_show
B1001,2026-10-01,2026-10-03,2
B1002,2026-10-01,2026-10-02,1
B1003,2026-10-02,2026-10-04,2
B1004,2026-10-02,2026-10-05,3
B1005,2026-10-03,2026-10-05,2
B1006,2026-10-03,2026-10-04,1
B1007,2026-10-04,2026-10-06,2
B1008,2026-10-04,2026-10-06,2
B1009,2026-10-05,2026-10-07,2
B1010,2026-10-05,2026-10-08,3


In [0]:
duplicates_df = bookings_df.groupBy("booking_id").count().filter(F.col("count") > 1)
display(duplicates_df)
deduped_bookings_df = bookings_sized_df.dropDuplicates(["booking_id"])

booking_id,count
B1006,2
B1021,2


In [0]:
windowSpec = Window.orderBy(F.col("total_net_revenue").desc())

movie_ranking_df = bookings_sized_df.filter(F.col("booking_status") == "Confirmed") \
    .groupBy("movie_title") \
    .agg(F.round(F.sum("net_amount"), 2).alias("total_net_revenue")) \
    .withColumn("rank", F.rank().over(windowSpec))

display(movie_ranking_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


movie_title,total_net_revenue,rank
The Last Island,5905.0,1
Starlight Express,5500.0,2
Code Red,3087.0,3
River of Dreams,2380.0,4


In [0]:
clean_joined_df = deduped_bookings_df.join(cinemas_df, on="cinema_id", how="left")

clean_joined_df.write.format("delta").mode("overwrite").saveAsTable("cinema_bookings_clean")

In [0]:
%sql
SELECT * FROM cinema_bookings_clean LIMIT 5;

cinema_id,booking_id,movie_title,booking_date,show_date,tickets,price_per_ticket,discount_pct,booking_status,payment_method,gross_amount,net_amount,booking_size,cinema_name,facilities,location,screen_count
C001,B1001,Starlight Express,2026-10-01,2026-10-03,2,250,10,Confirmed,UPI,500,450.0,Regular,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C001,B1007,Code Red,2026-10-04,2026-10-06,3,250,0,Confirmed,UPI,750,750.0,Regular,Galaxy Screens,"List(Parking, Food Court, Wheelchair Access)","List(Hyderabad, West)",5
C005,B1017,The Last Island,2026-10-09,2026-10-11,4,320,15,Refunded,Card,1280,1088.0,Large,Capital Cineplex,"List(Parking, Food Court, Wheelchair Access)","List(Delhi, East)",8
C006,B1018,Starlight Express,2026-10-09,2026-10-11,2,210,0,Confirmed,UPI,420,420.0,Regular,Orchid Screens,"List(Food Court, Wheelchair Access)","List(Pune, West)",4
C002,B1020,Code Red,2026-10-10,2026-10-13,3,300,10,Confirmed,Card,900,810.0,Regular,Silverlight Multiplex,"List(Parking, Food Court)","List(Bengaluru, North)",7


In [0]:
processed_bookings_df = bookings_df.dropDuplicates(["booking_id"]).fillna({"discount_pct": 0,
     "payment_method": "Unknown"}).withColumn("gross_amount", 
     F.col("tickets") * F.col("price_per_ticket")).withColumn("net_amount",
     F.round(F.col("gross_amount") * (1 - (F.col("discount_pct") / 100)), 2))

performance_df = processed_bookings_df.filter(F.col("booking_status") == "Confirmed").join(
        cinemas_df, on="cinema_id", how="left").groupBy(
        F.coalesce(F.col("location.city"), F.lit("Unknown")).alias("city"),
        "movie_title"
    ).agg(
        F.sum("tickets").alias("total_tickets_sold"),
        F.round(F.sum("net_amount"), 2).alias("total_confirmed_revenue"),
        F.count("booking_id").alias("confirmed_booking_count")
    ).orderBy("city", "movie_title")

performance_df.write.format("delta").mode("overwrite").saveAsTable("cinema_performance_report")

display(spark.table("cinema_performance_report"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


city,movie_title,total_tickets_sold,total_confirmed_revenue,confirmed_booking_count
Bengaluru,Code Red,3,810.0,1
Bengaluru,Starlight Express,2,600.0,1
Bengaluru,The Last Island,4,1200.0,2
Chennai,Code Red,3,513.0,1
Chennai,River of Dreams,4,720.0,1
Chennai,Starlight Express,4,684.0,1
Chennai,The Last Island,1,180.0,1
Delhi,Code Red,2,640.0,1
Delhi,River of Dreams,5,1440.0,1
Delhi,Starlight Express,6,1760.0,2
